# Experiment 5 — Naive Bayes: Text Classification

**Aim:** Build a Naive Bayes classifier that detects spam SMS messages, using text cleaning and TF-IDF vectorisation, and evaluate it with accuracy, precision and recall.

**Description:** Naive Bayes is a probabilistic classifier based on Bayes' theorem that assumes features are independent of each other. It works very well for text problems such as spam filtering. Text is first converted into numbers using Bag-of-Words or TF-IDF, and the classifier then estimates the probability of each class given the words in a message.

**Dataset:** SMS Spam Collection (local copy: `datasets/spam.csv`, from the [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms-spam-collection)).

In [1]:
import string
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42

# Locate project folders so the notebook runs from the project root or notebooks/
for candidate in [Path("datasets"), Path("../datasets")]:
    if candidate.exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError("datasets/ folder not found")

for candidate in [Path("outputs"), Path("../outputs")]:
    if candidate.exists():
        OUTPUT_DIR = candidate
        break

RES_DIR = OUTPUT_DIR / "results"
RES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the dataset

The CSV has a `v1` column with the label (`ham` / `spam`) and a `v2` column with the message text. The three `Unnamed` columns are leftover empty columns from the original file and are dropped.

In [2]:
df = pd.read_csv(DATA_DIR / "spam.csv", encoding="latin-1")
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})

print("Shape:", df.shape)
print("\nLabel counts:")
print(df["label"].value_counts())
df.head(3)

Shape: (5572, 2)

Label counts:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


## 2. Clean the text and split into train / test sets

Cleaning converts every message to lowercase and removes punctuation. The split is stratified so the spam/ham ratio is the same in both sets.

In [3]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text


df["cleaned"] = df["message"].apply(clean_text)
print("Example raw -> cleaned:")
print(" raw   :", df.loc[0, "message"][:70])
print(" clean :", df.loc[0, "cleaned"][:70])

X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned"], df["label"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"]
)
print(f"\nTrain size: {len(X_train)} | Test size: {len(X_test)}")

Example raw -> cleaned:
 raw   : Go until jurong point, crazy.. Available only in bugis n great world l
 clean : go until jurong point crazy available only in bugis n great world la e

Train size: 4457 | Test size: 1115


## 3. Convert messages to TF-IDF vectors

`TfidfVectorizer` is **fit on the training data only** and then used to transform the test data, so no information leaks from the test set.

In [4]:
vectorizer = TfidfVectorizer(stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Train matrix shape:", X_train_vec.shape)

Vocabulary size: 8126
Train matrix shape: (4457, 8126)


## 4. Train MultinomialNB and measure accuracy, precision and recall

Spam is treated as the positive class, so precision answers *"of the messages flagged as spam, how many really were spam?"* and recall answers *"of all real spam messages, how many did we catch?"*.

In [5]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label="spam")
rec = recall_score(y_test, y_pred, pos_label="spam")

print(f"Accuracy          : {acc:.4f}")
print(f"Precision (spam)  : {prec:.4f}")
print(f"Recall (spam)     : {rec:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix (rows = actual, cols = predicted, order = ham, spam):")
print(confusion_matrix(y_test, y_pred))

Accuracy          : 0.9623
Precision (spam)  : 1.0000
Recall (spam)     : 0.7181

Classification report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115

Confusion matrix (rows = actual, cols = predicted, order = ham, spam):
[[966   0]
 [ 42 107]]


## 5. Test the model on messages written by hand

Four new messages (a mix of obvious spam and ordinary texts) are cleaned and vectorised with the **same fitted vectorizer** before prediction.

In [6]:
my_messages = [
    "Congratulations! You have won a free ticket. Call now to claim your prize money",
    "Hey, are we still meeting for lunch tomorrow at 1 pm?",
    "URGENT: your mobile number has been selected for a 2000 cash prize. Text WIN to 80085 now",
    "Can you share the class notes from today's lecture when you get time",
]

my_vec = vectorizer.transform([clean_text(m) for m in my_messages])
my_pred = model.predict(my_vec)

print("Hand-written message test:")
for msg, label in zip(my_messages, my_pred):
    print(f"  [{label:>4}]  {msg}")

Hand-written message test:
  [spam]  Congratulations! You have won a free ticket. Call now to claim your prize money
  [ ham]  Hey, are we still meeting for lunch tomorrow at 1 pm?
  [spam]  URGENT: your mobile number has been selected for a 2000 cash prize. Text WIN to 80085 now
  [ ham]  Can you share the class notes from today's lecture when you get time


## Observations / Conclusion

Based on the run above:

1. The TF-IDF + MultinomialNB model reached **96.2% test accuracy** on 1,115 held-out messages, with a vocabulary of 8,126 terms.
2. **Precision for spam was 1.00** — the model never flagged a legitimate message as spam (confusion matrix: 0 ham → spam false positives). **Recall for spam was 0.72**, i.e. 42 of the 149 real spam messages were missed.
3. The dataset is imbalanced (4,825 ham vs 747 spam, ~87% ham), so accuracy alone hides the spam-miss rate. For a spam filter, **recall matters more** than precision here: a missed spam message is usually worse than a rare false alarm. Class weighting or a lower decision threshold could trade some precision for higher recall.
4. All **4 hand-written test messages** (2 obvious spam, 2 ordinary texts) were classified correctly, so the model generalises to fresh wording — not just the phrasing seen in training.